In [ ]:
from datasets import load_dataset

cve_ds = load_dataset("andstor/cvevc_cve")

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.59M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/420k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/463k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8963 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1366 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1607 [00:00<?, ? examples/s]

In [ ]:
patches_commits_ds = load_dataset("andstor/cvevc_commits", "patches")

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

In [ ]:
nonpatches_commits_ds = load_dataset("andstor/cvevc_commits", "non_patches", num_proc=20)

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

...

Generating test split:   0%|          | 0/2150904 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2485831 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/79 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/52 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/58 [00:00<?, ?it/s]

In [5]:
nonpatches_commits_ds

DatasetDict({
    train: Dataset({
        features: ['commit_id', 'repo', 'commit_message', 'diff', 'label'],
        num_rows: 3632163
    })
    test: Dataset({
        features: ['commit_id', 'repo', 'commit_message', 'diff', 'label'],
        num_rows: 2150904
    })
    validation: Dataset({
        features: ['commit_id', 'repo', 'commit_message', 'diff', 'label'],
        num_rows: 2485831
    })
})

In [6]:
from tqdm import tqdm

patch_cve = {} # from cve dataset

for split in cve_ds:
    print(split)
    for cve in tqdm(cve_ds[split]):
        commits = cve["commits"]
        cve = cve["cve"]
        for commit in commits:
            patch_cve[commit] = cve
        


train


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8963/8963 [00:00<00:00, 19332.54it/s]


test


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1366/1366 [00:00<00:00, 19536.60it/s]


validation


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1607/1607 [00:00<00:00, 19746.37it/s]


In [7]:
len(patch_cve)

15732

In [8]:
len(patches_commits_ds["train"]) + len(patches_commits_ds["validation"])+ len(patches_commits_ds["test"])

14526

In [9]:

from tqdm import tqdm

repo_cves = {}

for split in patches_commits_ds:
    print(split)
    for commit in tqdm(patches_commits_ds[split]):
        repo = commit["repo"]
        repo_cves.setdefault(repo, [])
        repo_cves[repo].append(patch_cve[commit["commit_id"]])
            
        

train


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11620/11620 [00:04<00:00, 2882.03it/s]


test


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1453/1453 [00:00<00:00, 10381.57it/s]


validation


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1453/1453 [00:00<00:00, 5840.41it/s]


In [10]:
from datasets import load_dataset, concatenate_datasets, DatasetDict, Dataset

# Create a new DatasetDict to hold the combined datasets
ds_commits = DatasetDict()
for split in nonpatches_commits_ds:
    ds_commits[split] = concatenate_datasets([nonpatches_commits_ds[split], patches_commits_ds[split]])


In [12]:
from datasets import DatasetDict
from tqdm import tqdm


ddict = DatasetDict()

for split in ds_commits:
    print(split)
    data = [] # {cve, commit_id, label}  
    for commit in tqdm(ds_commits[split]):
        repo = commit["repo"]
        cves = repo_cves[repo] # Get the CVEs for this repo
        commit_id = commit["commit_id"]
        
        patch = patch_cve.get(commit_id, None)  # Get the CVE for this commit, default to None if not found
        for cve in cves:
            label = 1 if cve == patch else 0  # Label is 1 if the CVE matches the patch, otherwise 0
            data.append({"cve": cve, "commit_id": commit_id, "label": label})

    ddict[split] = Dataset.from_list(data)

train


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3643783/3643783 [04:06<00:00, 14760.59it/s]


test


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2152357/2152357 [02:14<00:00, 15960.55it/s]


validation


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2487284/2487284 [02:36<00:00, 15920.81it/s]


In [129]:
ddict = ddict.sort(column_names=["cve", "commit_id"])

In [ ]:
ddict.push_to_hub("andstor/cvevc_cve_commit_mappings", private=False, max_shard_size="250MB")

Uploading the dataset shards:   0%|          | 0/14 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3537 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   0%|          |  530kB /  125MB            

...

Creating parquet from Arrow format:   0%|          | 0/2488 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   1%|          |  528kB /  104MB            

CommitInfo(commit_url='https://huggingface.co/datasets/andstor/cvevc_cve_commit_mappings/commit/71d336b5639529333a80a2b0aa48f1ae1af625f8', commit_message='Upload dataset', commit_description='', oid='71d336b5639529333a80a2b0aa48f1ae1af625f8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/andstor/cvevc_cve_commit_mappings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='andstor/cvevc_cve_commit_mappings'), pr_revision=None, pr_num=None)